# 02. 데이터 전처리 파이프라인

**목표:** 원시 데이터를 모델 학습에 적합한 형태로 변환하고 저장합니다.

전처리 단계:
1. 결측치 처리 (중앙값/최빈값 대체)
2. 이상치 처리 (IQR 클리핑)
3. 범주형 변수 인코딩 (LabelEncoder)
4. 수치형 변수 스케일링 (StandardScaler)
5. 학습/테스트 분할 (80:20)

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from teen_mind.data.loader import load_raw_data
from teen_mind.data.preprocessor import MentalHealthPreprocessor

print('라이브러리 로드 완료')

In [ ]:
import os
raw_files = [f for f in os.listdir('../data/raw') if f.endswith('.csv')]
df = load_raw_data(raw_files[0])
print(f'원본 데이터: {df.shape}')
df.head()

## 1. 전처리기 학습 및 변환

In [ ]:
preprocessor = MentalHealthPreprocessor(scaler_type='standard', test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = preprocessor.fit_transform(df)

print(f'학습 데이터: {X_train.shape}')
print(f'테스트 데이터: {X_test.shape}')
print(f'클래스 분포 (학습): {pd.Series(y_train).value_counts().to_dict()}')
print(f'클래스 분포 (테스트): {pd.Series(y_test).value_counts().to_dict()}')

## 2. 전처리 전후 비교

In [ ]:
# 수치형 피처 스케일링 전후 비교
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

numeric_cols = ['daily_social_media_hours', 'sleep_hours', 'anxiety_level', 'stress_level']
df[numeric_cols].hist(ax=axes[0], bins=20)
axes[0].set_title('전처리 전 (원본 스케일)')

pd.DataFrame(X_train[:, :4], columns=numeric_cols[:4]).hist(ax=axes[1], bins=20)
axes[1].set_title('전처리 후 (표준화)')

plt.tight_layout()
plt.show()

## 3. 전처리기 저장

In [ ]:
import os
os.makedirs('../models/saved', exist_ok=True)

preprocessor.save('../models/saved/preprocessor.pkl')
preprocessor.save_processed_data()
print('전처리기 저장 완료!')